# 02_risk_calibration.ipynb - Calibración de Csl y Verificación de BP

**Metodología**: `metodologia.pdf` Secciones 5 y 6

Este notebook implementa la calibración de riesgo en DOS PASOS SEPARADOS:

1. **Paso 1**: Calibración de Csl usando la MEDIANA de caídas implícitas (filtradas por precio + momentum alcista)
2. **Paso 2**: Verificación INDEPENDIENTE del Buying Power usando el Csl YA FIJO

> **IMPORTANTE**: Csl NUNCA se optimiza para lograr un BP objetivo. El BP se verifica DESPUÉS como filtro de admisión separado.

## Importaciones

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Ruta al repo
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Importar funciones implementadas
from src.risk import find_optimal_coef_sl, calculate_buying_power_distribution
from src.data import load_ohlc_from_yfinance

ModuleNotFoundError: No module named 'src'

## Cargar Datos

**Pendiente de ejecución con datos reales** - Reemplaza el ticker y el periodo según necesites.

In [ ]:
# TODO: Ejecutar con datos reales cuando se tenga acceso a yfinance
# ticker = 'ABSI'  # Ejemplo: small-cap
# df = load_ohlc_from_yfinance(ticker, period='max', interval='1h')

# Para demostración, crear datos sintéticos mínimos:
np.random.seed(42)
n_bars = 2000
prices = 5 + 3 * np.cumsum(np.random.randn(n_bars)) / 100  # Precio pequeño con tendencia
df = pd.DataFrame({
    'Datetime': pd.date_range('2023-01-01', periods=n_bars, freq='H'),
    'Open': prices * (1 + np.random.randn(n_bars) * 0.001),
    'High': prices * (1 + np.abs(np.random.randn(n_bars)) * 0.002),
    'Low': prices * (1 - np.abs(np.random.randn(n_bars)) * 0.002),
    'Close': prices,
    'Volume': np.random.randint(1000, 10000, n_bars)
})
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').reset_index(drop=True)

print(f'Data loaded: {len(df)} bars')
print(f'Price range: ${df["Close"].min():.2f} - ${df["Close"].max():.2f}')
df.head()

---

## Paso 1: Calibración de Csl (Mediana)

Según `metodologia.pdf` Sección 5:
- Filtrar entradas por precio small-cap ($1-$20) y momentum alcista (ER > 0.3)
- Calcular coeficiente implícito por entrada: `Csl_i = (P_entry - P_min_20) / ATR`
- Tomar la **MEDIANA** de los coeficientes (no promedio, no búsqueda por BP)
- Aplicar filtro de admisión: mediana debe estar en [1.5, 3.0]

In [ ]:
c_sl = find_optimal_coef_sl(
    df=df,
    n_samples=500,
    lookforward_window=20,
    atr_period=50,
    price_min=1.0,
    price_max=20.0,
    momentum_k=10,
    momentum_threshold=0.3,
    admission_range=(1.5, 3.0),
    seed=42
)

if c_sl is None:
    print("⚠️  ACTIVO NO ADMITIDO: Datos insuficientes o Csl mediana fuera de rango [1.5, 3.0]")
else:
    print(f"✅ Csl calibrado (mediana): {c_sl:.3f}")

---

## Paso 2: Verificación de BP (Independiente)

Según `metodologia.pdf` Sección 6.2-6.3:
- Usar el Csl YA FIJO desde Paso 1
- Nueva muestra independiente de 500 entradas (semilla diferente: 7)
- Calcular distribución de Buying Power: `BP = P_entry * (risk_per_trade / (Csl * ATR))`
- Verificar: mediana(BP) ∈ [$100, $1800] Y max(BP) ≤ $3000

In [ ]:
if c_sl is not None:
    bp_result = calculate_buying_power_distribution(
        df=df,
        c_sl=c_sl,
        n_samples=500,
        lookforward_window=20,
        atr_period=50,
        price_min=1.0,
        price_max=20.0,
        risk_per_trade=100.0,
        seed=7  # Semilla distinta de find_optimal_coef_sl
    )
    
    print(f"Buying Power Distribution (n={bp_result['n_valid_samples']} samples):")
    print(f"  - Mediana: ${bp_result['median_bp']:.2f}")
    print(f"  - Máximo:  ${bp_result['max_bp']:.2f}")
    print(f"  - Admitido: {'✅ YES' if bp_result['admitted'] else '❌ NO'}")
    
    if bp_result['admitted']:
        print(f"\n✅ ASSET APTO PARA TRADING con Csl = {c_sl:.3f}")
    else:
        print(f"\n⚠️  ACTIVO NO ADMITIDO: BP fuera de rango aceptable")
else:
    print("⏭️  Saltando Paso 2: Csl no se pudo calibrar en Paso 1")

---

## Resumen del flujo

```
Paso 1: find_optimal_coef_sl(df, ...) -> Csl_mediana
  ├─ Filtra: precio $1-$20 + ER > 0.3 (momentum alcista)
  ├─ Muestra: 500 entradas aleatorias
  ├─ Calcula: Csl_i = (P_entry - P_min_20) / ATR para cada entrada
  ├─ Toma: MEDIANA de Csl_i
  └─ Aprueba: si Csl_mediana ∈ [1.5, 3.0]

Paso 2: calculate_buying_power_distribution(df, c_sl=Csl_del_Paso1, ...
  ├─ Usa: Csl FIJO desde Paso 1 (NO optimizar)
  ├─ Muestra: 500 entradas NUEVAS (semilla diferente)
  ├─ Calcula: BP_i = P_entry * (risk_per_trade / (Csl * ATR))
  ├─ Verifica: mediana(BP) ∈ [$100, $1800] AND max(BP) ≤ $3000
  └─ Aprueba: si ambos criterios se cumplen
```


---

## Notas de implementación

1. **Ningún bucle de optimización BP**: Se eliminó completamente la "Fase 2" vieja que ordenaba Csl y buscaba el primero que cumpliera BP objetivo.

2. **Dos pasos separados**: Csl se calcula PRIMERO como mediana (sin mirar BP), luego BP se verifica DESPUÉS como filtro independiente.

3. **Semillas diferentes**: `find_optimal_coef_sl` usa `seed=42`, `calculate_buying_power_distribution` usa `seed=7` para muestras independientes.

4. **Admisión estricta**: Si cualquier paso falla, el activo se descarta (retorna None o admitted=False).